# 14 - Train an Episode/Task DQN Model Offline

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, but uses **two** action-value heads:

- **Episode head** (`action_value_episode`) predicts remaining reward in the current episode (stepwise Bellman; no bootstrap into the next episode).
- **Task head** (`action_value_task`) predicts reward in subsequent episodes. Its λ-return skips intra-episode steps and bootstraps from the next episode start.

Actions maximize the **sum** of the two Q-values. Both Bellman targets share one delayed next action `a* = argmax_a (Q_e + Q_t)`, so the heads cannot bootstrap from different actions. There is no intrinsic reward and no separate exploration policy.

The hypothesis is that this shortens the path through which later-episode reward credits earlier exploratory actions, so the agent learns useful in-context exploration from task reward alone.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb` (`get_action` already sums the two heads).


In [ ]:
import torch

from mouse_core import AdamW, AdamWFp32
from mouse_core.data import (
    DataLoader,
    Augmenter,
    NumericTokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import EpisodeTaskDqnObjective
from mouse_core.models import Model, Polyak, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"                   # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-episode-task-offline"  # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                                        # number of discrete actions predicted by each head
MAX_OBS_DISCRETE = 64                                  # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                                  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                         # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU = 0.0005                           # delayed Q-head interpolation (0 = frozen, 1 = copy of online heads)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream. Multi-episode tasks in those streams are what the task head learns from.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)


## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `09_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = NumericTokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
        {
            "type": "learnable",
            "output_field": "value",
            "tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
)


## Build The Model

A Mouse Core `Model` has three main pieces:

- `NumericEmbedder` maps a tokenized `TokenBatch` (modalities keyed by name; add `vocab_size` / `std` here; `fourier` / `continuous` also need `fourier_min` / `fourier_max`) into vectors.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- Two `DiscreteActionValueHead`s, passed together as `action_value_episode` and `action_value_task`. `get_action` reads `Q_episode + Q_task`.

The backbone exposes `hidden_dim`, and the embedder and heads use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and episode/task done codes.
- `fourier` for scalar numeric values such as rewards.
- `learnable` for the trailing `value` token (no step field; flagged `head_output: True` so Q is read from it).

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
            "positions": 1,
            "fourier_min": 0.01,
            "fourier_max": 10.0,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "learnable",
            "field": "value",
            "tokens": 1,
            "std": 0.02,
            "positions": 1,
        },
    ],
)

head_kwargs = dict(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads={
        "action_value_episode": DiscreteActionValueHead(**head_kwargs),
        "action_value_task": DiscreteActionValueHead(**head_kwargs),
    },
).train().to(device=device, dtype=preferred_dtype(device))
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data, predictions, delayed_predictions)` computes the episode/task DQN loss and metrics.
4. `AdamW` updates weights. Swap in `AdamWFp32` to keep fp32 master copies of bf16 encoder/backbone weights so updates smaller than a bf16 ULP still accumulate (heads are already float32).
5. Delayed Q comes from a heads-only copy: `delayed_model = model.delayed_copy(heads=True)`. Both Q heads are delayed — after the online forward, `delayed_model(last_hidden_state=out.last_hidden_state, head_output_indices=out.head_output_indices, hidden_states=out.hidden_states)` runs the delayed heads on the online last-layer states (no second encoder/backbone pass). `polyak.update(tau_heads=POLYAK_TAU)` interpolates the delayed heads toward the online heads after the optimizer step.

`EpisodeTaskDqnObjective` trains both heads from one delayed `a*`. The episode head is one-step TD on env reward with `gamma_episode_*=0`. The task head drops env reward, uses `task_td_lambda=1` so intra-episode steps skip to the next episode start, and bootstraps `Q_e(s', a*) + Q_t(s', a*)` there.


In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.0, betas=(0.9, 0.95), eps=1e-08)
delayed_model = model.delayed_copy(heads=True)
polyak = Polyak(model, delayed_model)
objective = EpisodeTaskDqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=0.0,
    gamma_episode_truncated=0.0,
    gamma_task_terminal=0.0,
    gamma_task_truncated=0.0,
    task_gamma_step=1.0,
    task_gamma_episode_terminal=1.0,
    task_gamma_episode_truncated=1.0,
    episode_td_lambda=0.0,
    task_td_lambda=1.0,
    grouping_field="task_index",
)

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW | AdamWFp32, objective: EpisodeTaskDqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_out = delayed_model(
                last_hidden_state=out.last_hidden_state,
                head_output_indices=out.head_output_indices,
                hidden_states=out.hidden_states,
            )
        loss, metrics = objective(objective_data.to(device), out.predictions, delayed_out.predictions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(tau_heads=POLYAK_TAU)
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(model=model, delayed_model=delayed_model, polyak=polyak, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q_e={metrics['q_episode_mean']:.3f}  q_t={metrics['q_task_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")
